In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
# import seaborn as sns
# import matplotlib.pyplot as plt

import ast
import re
import shutil


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [4]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(150, 11)


,id,template,placeholders,session,cam,constructed_filename,label,top1_pred,correct_top1,top5_preds,correct_top5
65,260922-144221,Piling [something] up,books,books_1,cam2,260922-144221_cam2.avi,48,114,False,"[103, 116, 144, 48, 114]",True
134,260922-151428,Tearing [something] into two pieces,paper,paper_1,cam2,260922-151428_cam2.avi,149,149,True,"[92, 150, 172, 52, 149]",True


In [5]:
# rename top1_label to top1_pred
result = result.rename(columns={'id':'video_id'})

del result['constructed_filename']

result.sample(3)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
32,260922-143234,Attaching [something] to [something],"magnet, magnet",magnet_1,cam2,1,99,False,"[100, 95, 58, 57, 99]",False
118,260922-151017,Pushing [something] with [something],"cup, phone",lady_2,cam1,101,101,True,"[42, 125, 19, 94, 101]",True
73,260922-144312,Piling [something] up,books,books_1,cam1,48,144,False,"[112, 116, 102, 48, 144]",True


In [6]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(150, 174)


NpzFile './predictions.npz' with keys: logits, probs, video_ids, labels

# main result

In [7]:

n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()

print(f"Number of samples: {n}")
print(f"Top-1 accuracy: {top1_acc:.4f}")
print(f"Top-5 accuracy: {top5_acc:.4f}")

Number of samples: 150
Top-1 accuracy: 0.3600
Top-5 accuracy: 0.6400


- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [8]:
result.groupby("cam")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
cam,
cam0,0.20
cam1,0.46
cam2,0.42


In [9]:
result.groupby(['cam']).agg(
    # count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,top1_acc,top5_acc
cam,,
cam0,0.20,0.42
cam1,0.46,0.82
cam2,0.42,0.68


In [10]:
result.groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Attaching [something] to [something],15,0.00,0.20
Moving [something] up,15,0.40,0.87
Piling [something] up,15,0.20,0.80
Plugging [something] into [something],15,0.87,0.93
Pushing [something] with [something],15,0.60,0.87
Putting [something] next to [something],15,0.27,0.60
Putting [something] on a surface,15,0.27,0.47
Putting [something] onto [something],15,0.20,0.47
Showing a photo of [something] to the camera,15,0.33,0.33


In [11]:
result[result.cam=='cam1'].groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Attaching [something] to [something],5,0.0,0.4
Moving [something] up,5,0.4,1.0
Piling [something] up,5,0.2,1.0
Plugging [something] into [something],5,1.0,1.0
Pushing [something] with [something],5,0.8,1.0
Putting [something] next to [something],5,0.4,0.6
Putting [something] on a surface,5,0.2,0.8
Putting [something] onto [something],5,0.2,0.4
Showing a photo of [something] to the camera,5,1.0,1.0


In [21]:
result[result.cam=='cam1'].groupby(['session']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
session,,,
books_1,5,0.2000,1.0000
books_2,5,0.2000,0.6000
lady_1,5,1.0000,1.0000
lady_2,5,0.2000,0.8000
magnet_1,5,0.0000,0.4000
magnet_2,5,0.4000,0.8000
paper_1,5,0.4000,1.0000
paper_2,5,0.2000,0.6000
plug_1,5,1.0000,1.0000


# Granular analysis

##### id2label

In [13]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [14]:
result.head(1)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
0,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam0,49,49,True,"[106, 46, 1, 50, 49]",True


In [15]:
df = result[['video_id',	'cam'	,'session'	,'placeholders','template',	'label'	,'top1_pred','correct_top1','correct_top5']]
print(df.shape)
df.head(2)

(150, 9)


,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5
0,260922-134214,cam0,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True
1,260922-134214,cam1,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True


In [16]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5,prob_0,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260922-134214,cam0,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True,0.000004,...,0.000317,0.000000,0.000000,0.000000,0.000000,0.000001,0.001027,0.000008,0.000072,0.000008
1,260922-134214,cam1,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True,0.000002,...,0.000162,0.000000,0.000000,0.000000,0.000000,0.000001,0.000822,0.000004,0.000014,0.000004


#### Main Analysis 


In [17]:
pd.set_option('display.float_format',  '{:,.4f}'.format)

In [18]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        # top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [19]:
df.session.unique()

array(['plug_1', 'plug_2', 'magnet_1', 'magnet_2', 'books_1', 'books_2',
       'lady_1', 'lady_2', 'paper_1', 'paper_2'], dtype=object)

In [20]:
avg_prob_in_diff_cases(df, session_type='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc                                     examples
0  plug_1       5     0.9556    0.0236   1.0000  260922-134214, 260922-134301, 260922-134333
1  plug_2       5     0.0001    0.0002   1.0000  260922-141236, 260922-141249, 260922-141308
==================== Videos ====================
        video_id   cam session                placeholders                               template label top1_pred correct_top1 correct_top5 prob_49 prob_68 prob_1 prob_50 prob_81
1  260922-134214  cam1  plug_1  plug, electrical extension  Plugging [something] into [something]    49        49         True         True  0.9804  0.0044 0.0040  0.0034  0.0027

Top 5 Probability Breakdown:
  prob_49  ->  0.98040  |  Plugging [something] into [something]
  prob_68  ->  0.00436  |  Pretending to pick [something] up
  prob_1   ->  0.00404  |  Attach

In [22]:
avg_prob_in_diff_cases(df, session_type='lady', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= lady =========================
label 124: Showing a photo of [something] to the camera
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc                                     examples
0  lady_1       5     0.6620    0.2761   1.0000  260922-150726, 260922-150740, 260922-150754
1  lady_2       5     0.0001    0.0002   0.2000  260922-150856, 260922-150915, 260922-150927
==================== Videos ====================
         video_id   cam session placeholders                                      template label top1_pred correct_top1 correct_top5 prob_124 prob_125 prob_128 prob_126 prob_129
91  260922-150726  cam1  lady_1         lady  Showing a photo of [something] to the camera   124       124         True         True   0.3146   0.2536   0.0615   0.0542   0.0417

Top 5 Probability Breakdown:
  prob_124 ->  0.31457  |  Showing a photo of [something] to the camera
  prob_125 ->  0.25359  |  Showing [something] behind [something]
  prob_128 ->  

In [23]:
avg_prob_in_diff_cases(df, session_type='books', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= books =========================
label 48: Piling [something] up
cams used: ['cam1']
   session  videos  prob_mean  prob_std  top1acc                                     examples
0  books_1       5     0.3288    0.1066   0.2000  260922-144210, 260922-144221, 260922-144233
1  books_2       5     0.0000    0.0000   0.2000  260922-144349, 260922-144425, 260922-144448
==================== Videos ====================
         video_id   cam  session placeholders               template label top1_pred correct_top1 correct_top5 prob_48 prob_114 prob_144 prob_102 prob_116
61  260922-144210  cam1  books_1        books  Piling [something] up    48        48         True         True  0.4842   0.2142   0.1919   0.0394   0.0176

Top 5 Probability Breakdown:
  prob_48  ->  0.48423  |  Piling [something] up
  prob_114 ->  0.21424  |  Putting [something similar to other things that are already on the table]
  prob_144 ->  0.19190  |  Stacking [number of] [something]
  prob_10

In [24]:
avg_prob_in_diff_cases(df, session_type='magnet', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= magnet =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
    session  videos  prob_mean  prob_std  top1acc                                     examples
0  magnet_1       5     0.0211    0.0252   0.0000  260922-143234, 260922-143255, 260922-143323
1  magnet_2       5     0.0006    0.0011   0.4000  260922-143515, 260922-143535, 260922-143557
==================== Videos ====================
         video_id   cam   session    placeholders                              template label top1_pred correct_top1 correct_top5 prob_101 prob_42 prob_15 prob_131 prob_100
31  260922-143234  cam1  magnet_1  magnet, magnet  Attaching [something] to [something]     1       101        False        False   0.2630  0.2184  0.1134   0.0819   0.0530

Top 5 Probability Breakdown:
  prob_101 ->  0.26304  |  Pushing [something] with [something]
  prob_42  ->  0.21841  |  Moving [something] closer to [something]
  prob_15  ->  0.11339  |  Hittin

In [25]:
avg_prob_in_diff_cases(df, session_type='paper', label='Tearing [something] into two pieces', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= paper =========================
label 149: Tearing [something] into two pieces
cams used: ['cam1']
   session  videos  prob_mean  prob_std  top1acc                                     examples
0  paper_1       5     0.2421    0.2023   0.4000  260922-151251, 260922-151331, 260922-151352
1  paper_2       5     0.0000    0.0000   0.2000  260922-151521, 260922-151539, 260922-151607
==================== Videos ====================
          video_id   cam  session placeholders                             template label top1_pred correct_top1 correct_top5 prob_150 prob_149 prob_92 prob_172 prob_65
121  260922-151251  cam1  paper_1        paper  Tearing [something] into two pieces   149       150        False         True   0.8051   0.0773  0.0470   0.0157  0.0115

Top 5 Probability Breakdown:
  prob_150 ->  0.80508  |  Tearing [something] just a little bit
  prob_149 ->  0.07730  |  Tearing [something] into two pieces
  prob_92  ->  0.04697  |  Pulling two ends of [